# Crawl Tiktok data: Video + Comments

# 1. Set up environment

## 1.1 Install dependencies
- If you run this notebook on local, you only need to install yt-dlp

In [42]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [43]:
# Import libraries
import os
import requests
import http.cookiejar
import datetime
from datetime import datetime
import re
import json
import time
import pandas as pd
import xlsxwriter
import yt_dlp

In [44]:


cookie_file = "../Crawdata/www.tiktok.com_cookies.txt"

cj = http.cookiejar.MozillaCookieJar(cookie_file)

try:
  # Must have ignore_discard and ignore_expires to install entire cookie file
  cj.load(ignore_discard=True, ignore_expires=True)
  print("Successfully loaded cookie file")
except FileNotFoundError:
  print("Cookie file not found")
except Exception as e:
  print(f"An error occurred: {e}")

# Create fake header: Due to Tiktok may block request without user-agent
headers = {
  'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

url = "https://www.tiktok.com"
response = requests.get(url, cookies=cj, headers=headers)

# Check result
if response.status_code == 200:
  print("Successfully sent request with cookies")
  print(response.text[:500])  # Print first 500 chars of response
else:
  print("An error occurred, status code:", response.status_code)

Successfully loaded cookie file
Successfully sent request with cookies
<!doctype html> <html lang="en">   <head>     <link rel="icon" href="data:;base64,=" />     <script id="slardar-config" type="application/json">       {         "slardarClient": "SlardarWAF",         "sdkUrl": "https://sf16-website-login.neutral.ttwstatic.com/obj/tiktok_web_login_static/slardar/fe/sdk-web/browser.sg.js",         "bid": "slardar_us_waf",         "pid": "js-44",         "pluginPathPrefix": "https://sf16-website-login.neutral.ttwstatic.com/obj/tiktok_web_login_static/slardar/fe/sdk


In [45]:
# Create session with cookies and headers
session = requests.Session()
session.cookies = cj
session.headers.update(headers)
print("✅ Session created with cookies and headers")

✅ Session created with cookies and headers


## 1.3 Set up directories

In [46]:
# Define BASE_DIR if not already set
BASE_DIR = globals().get("BASE_DIR", os.getcwd())
BASE_DIR = r"D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results"

if not os.path.exists(BASE_DIR):
    os.makedirs(BASE_DIR)
    print(f"📁 Đã setup thư mục mới tại: {BASE_DIR}")
else:
    print(f"✅ Thư mục đã sẵn sàng tại: {BASE_DIR}")

# (Tùy chọn) In ra đường dẫn tuyệt đối để bạn dễ dàng biết chính xác file đang lưu ở đâu trên máy
print("Absolute Output folder:", os.path.abspath(BASE_DIR))

✅ Thư mục đã sẵn sàng tại: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results
Absolute Output folder: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results


# 2. Helper function 
This cell includes helper functions:
1. fetch_oembed:
> - oEmbed is a format for allowing an embedded representation of a URL on third party sites. It allows you to fetch metadata (title, author, description, etc.) of a video by its URL, which is useful for our crawling process

2. extract_aweme_id_from_url:
> - This function extracts the unique identifier (aweme_id) from a TikTok video URL, which is necessary for fetching specific video details.

In [47]:
def fetch_oembed(video_url, session=None, timeout=12):
  "Get metadata oEmbed of video"
  if session is None:
    session = globals().get("session", requests.Session())
  oembed_url = "https://www.tiktok.com/oembed"
  try:
    r = session.get(oembed_url, params={"url": video_url}, timeout=timeout)
    if r.status_code == 200:
      return r.json()
    return None
  except Exception as e:
    print(f"Error fetching oEmbed for {video_url}: {e}")
    return None

def extract_aweme_id_from_url(video_url):
  "Parse aweme_id from URL like /video/<digits>"
  m = re.search(r"/video/(\d+)", video_url)
  return m.group(1) if m else None


# 3. Crawl data
- In this part, we will crawl data by video URLs. You can change the list of URLs to crawl different videos.


##  3.1 Download video

In [48]:
def download_video(video_url, out_dir, write_info_json=True, max_filesize=None):
  os.makedirs(out_dir, exist_ok=True)
  out_template = os.path.join(out_dir, "%(id)s.%(ext)s")

  ydl_opts = {
    "outtmpl": out_template,
    "format": "best",
    "noplaylist": True,
    "cookiesfile": {
      "User-Agent": "Mozilla/5.0",
      "Referer": video_url
    },
    "download_archive": os.path.join(out_dir, "downloaded.txt"),
  }

  if write_info_json:
    ydl_opts.update({
      "writedescriptions": True,
      "writesubtitles": True,
      "writeinfojson": False,
    })
  if max_filesize:
    ydl_opts["max_filesize"] = max_filesize
  with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(video_url, download=True)

  # Get info json
  info_json_path = None
  if isinstance(info, dict) and "id" in info:
    candidate = os.path.join(out_dir, f"{info['id']}.info.json")
    if os.path.exists(candidate):
      info_json_path = candidate
  
  if info_json_path:
    with open(info_json_path, "r", encoding="utf-8") as f:
      info_json = json.load(f)
  else:
    info_data = info
  return info_data


## 3.2 Crawl comments

In [49]:
def fetch_comments_web(aweme_id: str, session: requests.Session, max_comments=200, sleep=1.0):
    url = "https://www.tiktok.com/api/comment/list/"
    cursor, out = 0, []
    while len(out) < max_comments:
        params = {"aid": 1988, "aweme_id": aweme_id, "cursor": cursor, "count": 20}
        r = session.get(url, params=params, timeout=20)
        if r.status_code != 200:
            print("HTTP", r.status_code, r.text[:200])
            break
        try:
            data = r.json()
        except Exception as e:
            print("JSON parse error:", e, r.text[:200]); break

        out.extend(data.get("comments") or [])
        print(f"Fetched {len(out)} comments so far…")

        if not data.get("has_more"):
            break
        cursor = data.get("cursor", cursor + 20)
        if sleep: time.sleep(sleep)
    return out


def fetch_replies_web(aweme_id: str, comment_id: str, session: requests.Session, max_replies=100, sleep=1.0):
    url = "https://www.tiktok.com/api/comment/list/reply/"
    cursor, out = 0, []
    while len(out) < max_replies:
        params = {"aid": 1988, "aweme_id": aweme_id, "comment_id": comment_id, "cursor": cursor, "count": 20}
        r = session.get(url, params=params, timeout=20)
        if r.status_code != 200:
            print("HTTP", r.status_code, r.text[:200])
            break
        try:
            data = r.json()
        except Exception as e:
            print("JSON parse error:", e, r.text[:200]); break

        out.extend(data.get("comments") or [])
        if not data.get("has_more"):
            break
        cursor = data.get("cursor", cursor + 20)
        if sleep: time.sleep(sleep)
    return out

def fetch_all_comments_with_replies_web(aweme_id: str, session: requests.Session,
                                        max_comments=200, max_replies_per_comment=50, sleep=1.0):
    comments = fetch_comments_web(aweme_id, session, max_comments=max_comments, sleep=sleep)
    results = []
    for c in comments:
        item = {
            "cid": c.get("cid"),
            "text": c.get("text"),
            "author": (c.get("user") or {}).get("nickname"),
            "create_time": c.get("create_time"),
            "like_count": c.get("digg_count"),
            "reply_count": c.get("reply_comment_total"),
            "replies": []
        }
        try:
            if item["cid"]:
                rs = fetch_replies_web(aweme_id, item["cid"], session, max_replies=max_replies_per_comment, sleep=sleep)
                item["replies"] = rs
        except Exception as e:
            print(f"⚠️ Reply error for {item['cid']}: {e}")
        results.append(item)
    return results

## 3.3 Crawl 1 video function

In [50]:
def crawl_one_tiktok(video_url, out_dir, session=None, use_comments=False, max_comments=200, max_replies_per_comment=50, sleep=1.0, download_video_file=True):
    if session is None:
        session = globals().get("session", requests.Session())
    
    os.makedirs(out_dir, exist_ok=True)
    result = {
        "url": video_url,
        "crawled_at": datetime.utcnow().isoformat() + "Z"
    }
    # 1) oEmbed

    if download_video_file:
        try:
            video_info = download_video(video_url, out_dir, write_info_json=True)
            result["video_info"] = video_info
            print("✅ Video downloaded successfully")
        except Exception as e:
            print(f"⚠️ Video download error: {e}")
            result["video_info"] = None

    # 2) oEmbed
    oembed = fetch_oembed(video_url, session=session)
    result["oembed"] = oembed

    # 3) Lấy aweme_id
    aweme_id = None
    if oembed and isinstance(oembed, dict):
        aweme_id = oembed.get("embed_product_id")
    if not aweme_id:
        aweme_id = extract_aweme_id_from_url(video_url)

    if not aweme_id:
        raise ValueError("Không lấy được aweme_id cho video này")

    print("aweme_id:", aweme_id)

    # 4) Comments (tuỳ chọn)
    if use_comments:
        cmts = fetch_all_comments_with_replies_web(
            aweme_id, session, max_comments=max_comments,
            max_replies_per_comment=max_replies_per_comment, sleep=sleep
        )
        result["comments"] = cmts

    # 5) Save metadata JSON
    json_path = os.path.join(out_dir, f"{aweme_id}_crawl.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    return result, json_path

# 4. Export excel file

In [51]:
def export_comments_to_excel(all_items, aweme_id: str, out_dir: str):
    import os, pandas as pd
    os.makedirs(out_dir, exist_ok=True)

    rows = []
    for c in all_items or []:
        # Top-level
        rows.append({
            "video_id": aweme_id,
            "cid": c.get("cid"),
            "parent_cid": None,
            "is_reply": 0,
            "author_name": c.get("author"),
            "text": c.get("text"),
            "like_count": c.get("like_count"),
            "reply_count": c.get("reply_count"),
            "create_time": c.get("create_time"),
        })
        # Replies
        for r in (c.get("replies") or []):
            ru = r.get("user") or {}
            rows.append({
                "video_id": aweme_id,
                "cid": r.get("cid"),
                "parent_cid": c.get("cid"),
                "is_reply": 1,
                "author_name": ru.get("nickname"),
                "text": r.get("text"),
                "like_count": r.get("digg_count") or r.get("like_count"),
                "reply_count": r.get("reply_comment_total") or r.get("reply_count"),
                "create_time": r.get("create_time"),
            })

    df = pd.DataFrame(rows)

    # Epoch -> datetime (tz-aware), rồi bỏ tz để Excel chấp nhận
    if "create_time" in df.columns:
        dt_utc = pd.to_datetime(df["create_time"], unit="s", errors="coerce", utc=True)
        # BẢN NAIVE (không timezone)
        df["created_at_utc"] = dt_utc.dt.tz_localize(None)
        df["created_at_vn"]  = dt_utc.dt.tz_convert("Asia/Ho_Chi_Minh").dt.tz_localize(None)

    # Sắp cột
    cols = ["video_id","cid","parent_cid","is_reply","author_name",
            "text","like_count","reply_count","create_time",
            "created_at_utc","created_at_vn"]
    df = df[[c for c in cols if c in df.columns]]

    xlsx_path = os.path.join(out_dir, f"{aweme_id}_comments.xlsx")
    with pd.ExcelWriter(xlsx_path, engine="xlsxwriter",
                        datetime_format="yyyy-mm-dd hh:mm:ss") as writer:
        df.to_excel(writer, index=False, sheet_name="comments")
        ws = writer.sheets["comments"]
        ws.freeze_panes(1, 0)
        # Auto-width
        for i, col in enumerate(df.columns):
            maxlen = min(60, max(10, df[col].astype(str).str.len().max() if not df.empty else 10))
            ws.set_column(i, i, maxlen + 2)

    print("✅ Saved Excel:", xlsx_path)
    return xlsx_path, df

# 5. Test on 1 video

In [52]:
# VIDEO_URL = "https://www.tiktok.com/@hesti.home/video/7628593511155109140"  

# res, jsonp = crawl_one_tiktok(
#     VIDEO_URL, BASE_DIR,
#     use_comments=True,         # bật/tắt crawl bình luận
#     max_comments=100,          # tối đa số bình luận cấp 1
#     max_replies_per_comment=50 # tối đa số reply/1 bình luận
# )
# print("Metadata saved to:", jsonp)
# print("Keys:", list(res.keys()))

# # Nếu có comments thì xuất Excel
# cmts = res.get("comments", [])
# if cmts:
#     aweme_id = extract_aweme_id_from_url(VIDEO_URL) or (res.get("oembed") or {}).get("embed_product_id")
#     xlsx_path, df_preview = export_comments_to_excel(cmts, aweme_id, BASE_DIR)
#     print(df_preview.head(3))
# else:
#     print("No comments fetched.")


# 6. Crawl multiple videos

In [53]:
json_file_path = "../Crawdata/dataset_tiktok-scraper_2026-04-17_10-26-02-949.json"

try:
  with open(json_file_path, "r", encoding="utf-8") as f:
    videos_data = json.load(f)
  print(f"Successfully loaded JSON data from {json_file_path}, total videos: {len(videos_data)}")
except FileNotFoundError:
  print(f"❌ JSON file not found at: {json_file_path}")
  videos_data = []

results_summary = []
for idx, video_info in enumerate(videos_data, 1):
  try:
    video_url = video_info.get("webVideoUrl")
    if not video_url:
      print(f"Skipping video #{idx} due to missing URL")
      continue
    print(f"\n{'='*60}")
    print(f"Video {idx}/{len(videos_data)}: {video_url}")
    print(f"\n{'='*60}")

    video_dir = os.path.join(BASE_DIR, f"video_{idx}")

    res, jsonp = crawl_one_tiktok(
      video_url,
      video_dir,
      use_comments=True,
      max_comments=50,
      max_replies_per_comment=20,
      download_video_file=True
    )
    cmts = res.get("comments", [])
    if cmts:
        aweme_id = extract_aweme_id_from_url(video_url) or (res.get("oembed") or {}).get("embed_product_id")
        xlsx_path, df_preview = export_comments_to_excel(cmts, aweme_id, video_dir)
        print(df_preview.head(3))
    else:
        print("No comments fetched.")


    results_summary.append({
      "index": idx,
      "url": video_url,
      "author": video_info.get("authorMeta.name"),
      "status": "success",
      "json_path": jsonp,
    })
    print(f"Video {idx} crawl successful")
  except Exception as e:
    print(f"❌ Error crawling video {idx}: {e}")
    results_summary.append({
      "index": idx,
      "url": video_info.get("webVideoUrl", "Unknown"),
      "author": video_info.get("authorMeta.name", "Unknown"),
      "status": f"error: {str({e})}",
    })

print(f"\n{'='*60}")
print("SUMMARY RESULTS:")
print(f"\n{'='*60}")

for item in results_summary:
  print(f"Video #{item['index']}: {item['url']} - Author: {item['author']} - Status: {item['status']}")
print(f"\nTotal videos processed: {len(results_summary)}")


Successfully loaded JSON data from ../Crawdata/dataset_tiktok-scraper_2026-04-17_10-26-02-949.json, total videos: 26

Video 1/26: https://www.tiktok.com/@cahebietnhayy/video/7629295002790956308

[TikTok] Extracting URL: https://www.tiktok.com/@cahebietnhayy/video/7629295002790956308
[TikTok] 7629295002790956308: Downloading webpage


C:\Users\KietKlat\AppData\Local\Temp\ipykernel_27184\2560448234.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "crawled_at": datetime.utcnow().isoformat() + "Z"


[TikTok] Solving JS challenge using native Python implementation
[TikTok] 7629295002790956308: Downloading webpage with challenge cookie
[info] 7629295002790956308: Downloading 1 format(s): bytevc1_1080p_897128-1
[info] There are no subtitles for the requested languages
[download] Destination: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_1\7629295002790956308.mp4
[download] 100% of    2.15MiB in 00:00:01 at 1.69MiB/s   
✅ Video downloaded successfully
aweme_id: 7629295002790956308
Fetched 20 comments so far…
Fetched 40 comments so far…
Fetched 60 comments so far…
✅ Saved Excel: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_1\7629295002790956308_comments.xlsx
              video_id                  cid           parent_cid  is_reply  \
0  7629295002790956308  7629306018900869908                  NaN         0   
1  7629295002790956308  7629308783604613905  7629306018900869908         1   
2  7629295002790956308  7629360045633405703  76293

C:\Users\KietKlat\AppData\Local\Temp\ipykernel_27184\2560448234.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "crawled_at": datetime.utcnow().isoformat() + "Z"


[TikTok] Solving JS challenge using native Python implementation
[TikTok] 7629213074599316744: Downloading webpage with challenge cookie
[info] 7629213074599316744: Downloading 1 format(s): bytevc1_1080p_397450-1
[info] There are no subtitles for the requested languages
[download] Destination: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_2\7629213074599316744.mp4
[download] 100% of    1.62MiB in 00:00:00 at 1.94MiB/s   
✅ Video downloaded successfully
aweme_id: 7629213074599316744
Fetched 3 comments so far…
✅ Saved Excel: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_2\7629213074599316744_comments.xlsx
              video_id                  cid           parent_cid  is_reply  \
0  7629213074599316744  7629326366093312776                  NaN         0   
1  7629213074599316744  7629509285769069330  7629326366093312776         1   
2  7629213074599316744  7629376778641163016                  NaN         0   

  author_name               

...

[TikTok] Solving JS challenge using native Python implementation
[TikTok] 7626061941823245586: Downloading webpage with challenge cookie
[info] 7626061941823245586: Downloading 1 format(s): bytevc1_1080p_1423463-1
[info] There are no subtitles for the requested languages
[download] Destination: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_24\7626061941823245586.mp4
[download] 100% of    2.53MiB in 00:00:01 at 1.87MiB/s   
✅ Video downloaded successfully
aweme_id: 7626061941823245586
Fetched 20 comments so far…
Fetched 40 comments so far…
Fetched 60 comments so far…
✅ Saved Excel: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_24\7626061941823245586_comments.xlsx
              video_id                  cid           parent_cid  is_reply  \
0  7626061941823245586  7626419248260416263                  NaN         0   
1  7626061941823245586  7628895068380168978  7626419248260416263         1   
2  7626061941823245586  7626947202472805136  76

C:\Users\KietKlat\AppData\Local\Temp\ipykernel_27184\2560448234.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "crawled_at": datetime.utcnow().isoformat() + "Z"


[TikTok] Solving JS challenge using native Python implementation
[TikTok] 7629193655890234645: Downloading webpage with challenge cookie
[info] 7629193655890234645: Downloading 1 format(s): bytevc1_1080p_1085524-1
[info] There are no subtitles for the requested languages
[download] Destination: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_25\7629193655890234645.mp4
[download] 100% of    1.51MiB in 00:00:00 at 1.81MiB/s   
✅ Video downloaded successfully
aweme_id: 7629193655890234645
Fetched 9 comments so far…
✅ Saved Excel: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_25\7629193655890234645_comments.xlsx
              video_id                  cid parent_cid  is_reply  \
0  7629193655890234645  7629296367671952136        NaN         0   
1  7629193655890234645  7629579813276025617        NaN         0   
2  7629193655890234645  7629623858631459600        NaN         0   

       author_name                                            tex

C:\Users\KietKlat\AppData\Local\Temp\ipykernel_27184\2560448234.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "crawled_at": datetime.utcnow().isoformat() + "Z"


[TikTok] Solving JS challenge using native Python implementation
[TikTok] 7628434758980144404: Downloading webpage with challenge cookie
[info] 7628434758980144404: Downloading 1 format(s): bytevc1_540p_594266-1
[info] There are no subtitles for the requested languages
[download] Destination: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_26\7628434758980144404.mp4
[download] 100% of    1.07MiB in 00:00:00 at 1.91MiB/s   
✅ Video downloaded successfully
aweme_id: 7628434758980144404
Fetched 1 comments so far…
✅ Saved Excel: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_26\7628434758980144404_comments.xlsx
              video_id                  cid           parent_cid  is_reply  \
0  7628434758980144404  7628929949945004820                  NaN         0   
1  7628434758980144404  7628931639079830293  7628929949945004820         1   

    author_name text  like_count  reply_count  create_time  \
0  Edt5ty Hoàng  🥰🥰🥰         1.0          1